# AI and Social Media Impact on Student Health and Grades
### Interactive EDA and End-to-End Machine Learning

**Author:** Mobeen Fatima

This notebook explores the dataset attached in Kaggle Input. It includes automated file discovery, cleaning, missing-value analysis, interactive visualizations, feature engineering, model comparison, validation, feature importance, and CSV exports.

The code discovers the real column names and target at runtime rather than assuming a fixed schema. Check the selected target before interpreting model scores. Associations in student survey data do not establish causation, and health-related outputs are not clinical diagnoses.

## 1. Imports and configuration

Load data science, visualization, and machine-learning libraries. Plotly charts are interactive; Matplotlib is the fallback.

In [1]:
import os, re, glob, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",100)
try:
    import plotly.express as px
    PLOTLY_AVAILABLE=True
except Exception:
    PLOTLY_AVAILABLE=False
RANDOM_STATE=42
print("Plotly available:",PLOTLY_AVAILABLE)

Plotly available: True


## 2. Discover dataset files

Find CSV, Excel, and Parquet files mounted by Kaggle.

In [2]:
ROOT=Path("/kaggle/input")
files=sorted(set(p for pat in ["**/*.csv","**/*.xlsx","**/*.xls","**/*.parquet"] for p in ROOT.glob(pat) if p.is_file()))
print("Data files:",len(files))
for p in files: print(p, f"{p.stat().st_size/1024**2:.2f} MB")
if not files: raise FileNotFoundError("Attach the requested dataset in Kaggle Input.")

Data files: 1
/kaggle/input/datasets/srisyra02/ai-and-social-media-impact-student-health-and-grades/AI_SocialMedia_Student_Dataset.csv 0.93 MB


## 3. Load the most relevant file

Choose the most relevant filename and load the data without hard-coded filenames.

In [3]:
keys=["social","student","impact","health","grade","ai"]
files=sorted(files,key=lambda p:(sum(k in p.name.lower() for k in keys),p.stat().st_size),reverse=True)
DATA_PATH=files[0]
def read_data(p):
    if p.suffix.lower()==".csv":
        try: return pd.read_csv(p,low_memory=False)
        except UnicodeDecodeError: return pd.read_csv(p,encoding="latin-1",low_memory=False)
    if p.suffix.lower() in [".xlsx",".xls"]: return pd.read_excel(p)
    return pd.read_parquet(p)
df=read_data(DATA_PATH)
print("Selected:",DATA_PATH,"Shape:",df.shape)
display(df.head())

Selected: /kaggle/input/datasets/srisyra02/ai-and-social-media-impact-student-health-and-grades/AI_SocialMedia_Student_Dataset.csv Shape: (16000, 10)


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04


## 4. Dataset overview

Review dimensions, column names, and representative rows.

In [4]:
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"unique":df.nunique(dropna=True).values}).head(100))
display(df.head(10))

Rows: 16,000 | Columns: 10


,column,dtype,unique
0,Student_ID,object,16000
1,Age,int64,13
2,Gender,object,3
3,Education_Level,object,3
4,Daily_Social_Media_Hours,float64,1106
5,Daily_AI_Tool_Usage_Hours,float64,762
6,Sleep_Hours,float64,752
7,Physical_Activity_Hours,float64,462
8,Mental_Health_Score,float64,3743
9,Physical_Health_Score,float64,3433


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04
5,STU_00006,25,Female,University,6.04,3.67,6.40,0.00,59.82,78.03
6,STU_00007,17,Male,High School,1.98,1.66,7.03,4.44,83.05,97.97
7,STU_00008,19,Female,College,3.42,5.10,4.82,0.00,78.76,85.95
8,STU_00009,22,Female,University,7.85,0.70,7.12,0.40,72.60,82.53
9,STU_00010,15,Female,High School,6.96,1.60,5.78,1.67,71.83,91.23


## 5. Standardize column names

Normalize column names and retain a mapping for traceability.

In [5]:
original_cols=list(df.columns)
def norm_col(x):
    x=re.sub(r"[^A-Za-z0-9]+","_",str(x).strip()).strip("_")
    return re.sub(r"_+","_",x)
cols=[]; seen={}
for c in map(norm_col,df.columns):
    seen[c]=seen.get(c,0)+1
    cols.append(c if seen[c]==1 else f"{c}_{seen[c]}")
df.columns=cols
display(pd.DataFrame({"original":original_cols,"standardized":cols}))

,original,standardized
0,Student_ID,Student_ID
1,Age,Age
2,Gender,Gender
3,Education_Level,Education_Level
4,Daily_Social_Media_Hours,Daily_Social_Media_Hours
5,Daily_AI_Tool_Usage_Hours,Daily_AI_Tool_Usage_Hours
6,Sleep_Hours,Sleep_Hours
7,Physical_Activity_Hours,Physical_Activity_Hours
8,Mental_Health_Score,Mental_Health_Score
9,Physical_Health_Score,Physical_Health_Score


## 6. Data quality audit

Check missing values, duplicate records, unique values, and types.

In [6]:
quality=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),"missing_pct":(df.isna().mean()*100).round(2),"unique":df.nunique(dropna=True)}).sort_values("missing_pct",ascending=False)
print("Duplicate rows:",df.duplicated().sum())
display(quality)

Duplicate rows: 0


,dtype,missing,missing_pct,unique
Student_ID,object,0,0.0,16000
Age,int64,0,0.0,13
Gender,object,0,0.0,3
Education_Level,object,0,0.0,3
Daily_Social_Media_Hours,float64,0,0.0,1106
Daily_AI_Tool_Usage_Hours,float64,0,0.0,762
Sleep_Hours,float64,0,0.0,752
Physical_Activity_Hours,float64,0,0.0,462
Mental_Health_Score,float64,0,0.0,3743
Physical_Health_Score,float64,0,0.0,3433


## 7. Missing-value visualization

Visualize missingness with interactive Plotly bars.

In [7]:
m=df.isna().sum().sort_values(ascending=False); m=m[m>0]
if len(m):
    md=m.rename_axis("column").reset_index(name="missing_count"); md["missing_pct"]=md.missing_count/len(df)*100
    if PLOTLY_AVAILABLE: px.bar(md,x="column",y="missing_pct",hover_data=["missing_count"],title="Missing Values (%)").show()
    else: md.plot.bar(x="column",y="missing_pct",legend=False); plt.xticks(rotation=60); plt.tight_layout(); plt.show()
else: print("No missing values found.")

No missing values found.


## 8. Basic cleaning

Remove duplicate rows and completely empty columns; do not silently delete unusual values.

In [8]:
before=df.shape
df=df.drop_duplicates().copy()
empty=[c for c in df if df[c].isna().all()]
df=df.drop(columns=empty)
print("Before:",before,"After:",df.shape,"Removed empty columns:",empty)

Before: (16000, 10) After: (16000, 10) Removed empty columns: []


## 9. Separate variable types

Identify numerical and categorical features.

In [9]:
num_cols=df.select_dtypes(include=np.number).columns.tolist()
cat_cols=df.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric:",num_cols)
print("Categorical:",cat_cols)

Numeric: ['Age', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score']
Categorical: ['Student_ID', 'Gender', 'Education_Level']


## 10. Numerical summary

Review the statistical distribution and range of numeric columns.

In [10]:
if num_cols: display(df[num_cols].describe().T)
else: print("No numeric columns.")

,count,mean,std,min,25%,50%,75%,max
Age,16000.0,19.038750,3.765270,13.00,16.0000,19.000,22.00,25.00
Daily_Social_Media_Hours,16000.0,4.541791,2.403044,0.00,2.8400,4.500,6.18,14.00
Daily_AI_Tool_Usage_Hours,16000.0,2.590624,1.679376,0.00,1.3100,2.520,3.75,9.50
Sleep_Hours,16000.0,6.545371,1.270515,2.00,5.6800,6.540,7.41,11.15
Physical_Activity_Hours,16000.0,1.248359,1.038632,0.00,0.3100,1.130,1.96,5.00
Mental_Health_Score,16000.0,72.493193,9.244818,32.56,66.7600,73.750,79.55,91.76
Physical_Health_Score,16000.0,88.020465,9.691037,48.03,81.5175,89.455,97.17,99.98


## 11. Interactive numeric distributions

Create interactive histograms and box plots for numeric columns.

In [11]:
# Display distributions of numeric columns
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

if num_cols:
    for c in num_cols[:10]:
        # Convert values to numeric and remove missing/infinite values
        values = pd.to_numeric(df[c], errors="coerce")
        values = values[values.notna() & values.between(float("-inf"), float("inf"))]

        if values.empty:
            print(f"Skipping {c}: no valid numeric values.")
            continue

        # Try Plotly first; fall back to Matplotlib if rendering fails
        plotted = False

        if PLOTLY_AVAILABLE:
            try:
                fig = px.histogram(
                    x=values,
                    nbins=30,
                    marginal="box",
                    title=f"Distribution of {c}",
                    labels={"x": c, "y": "Frequency"},
                    template="plotly_white"
                )
                fig.update_layout(
                    width=900,
                    height=500,
                    showlegend=False
                )
                display(fig)
                plotted = True
            except Exception as e:
                print(f"Plotly rendering failed for {c}. Using Matplotlib.")

        if not plotted:
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.hist(values, bins=30, edgecolor="black")
            ax.set_title(f"Distribution of {c}")
            ax.set_xlabel(c)
            ax.set_ylabel("Frequency")
            ax.grid(axis="y", alpha=0.3)
            plt.tight_layout()
            display(fig)
            plt.close(fig)

    print(f"Displayed distributions for up to {min(10, len(num_cols))} numeric columns.")

else:
    print("No numeric columns to plot.")

Displayed distributions for up to 7 numeric columns.


## 12. Categorical exploration

Inspect frequent values in categorical variables.

In [12]:
# Display categorical distributions with tables and charts
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

if cat_cols:
    for c in cat_cols[:12]:
        counts = (
            df[c]
            .astype("string")
            .fillna("Missing")
            .value_counts()
            .head(15)
            .rename_axis(c)
            .reset_index(name="count")
        )

        print(f"\nCategory Distribution: {c}")
        display(counts)

        if counts.empty:
            print(f"No categories available for {c}.")
            continue

        plotted = False

        # Try Plotly first
        if PLOTLY_AVAILABLE:
            try:
                fig = px.bar(
                    counts,
                    x=c,
                    y="count",
                    title=f"Top Categories: {c}",
                    labels={c: c, "count": "Frequency"},
                    template="plotly_white",
                    text="count"
                )

                fig.update_layout(
                    width=900,
                    height=500,
                    xaxis_tickangle=-45,
                    showlegend=False
                )
                fig.update_traces(textposition="outside")
                display(fig)
                plotted = True

            except Exception as e:
                print(f"Plotly rendering failed for {c}. Using Matplotlib.")

        # Fallback to Matplotlib
        if not plotted:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.bar(counts[c].astype(str), counts["count"])
            ax.set_title(f"Top Categories: {c}")
            ax.set_xlabel(c)
            ax.set_ylabel("Frequency")
            ax.tick_params(axis="x", rotation=45)
            plt.tight_layout()
            display(fig)
            plt.close(fig)

else:
    print("No categorical columns available to plot.")


Category Distribution: Student_ID


,Student_ID,count
0,STU_15984,1
1,STU_15983,1
2,STU_15982,1
3,STU_15981,1
4,STU_15980,1
5,STU_15979,1
6,STU_15978,1
7,STU_15977,1
8,STU_15976,1
9,STU_15975,1



Category Distribution: Gender


,Gender,count
0,Male,7744
1,Female,7628
2,Non-binary,628



Category Distribution: Education_Level


,Education_Level,count
0,High School,6083
1,University,5027
2,College,4890


## 13. Target discovery

Search common target names for social impact, health, grades, and academic outcomes.

In [13]:
target_names=["Overall_Impact","Impact_Level","Academic_Performance","Academic_Impact","Grade","GPA","Mental_Health_Index","Stress_Level","Sleep_Quality"]
lookup={c.lower():c for c in df.columns}
matches=[lookup[x.lower()] for x in target_names if x.lower() in lookup]
print("Potential targets:",matches)
print("Review the dataset's documentation before accepting an automatically selected target.")

Potential targets: []
Review the dataset's documentation before accepting an automatically selected target.


## 14. Set target and infer task

Set TARGET_COLUMN manually if needed; the notebook otherwise tries to identify a likely outcome.

In [14]:
TARGET_COLUMN=None  # Example: "Overall_Impact"
if TARGET_COLUMN is None:
    preferred=["overall_impact","impact_level","academic_performance","academic_impact","grade","gpa"]
    TARGET_COLUMN=next((lookup[x] for x in preferred if x in lookup),matches[0] if matches else None)
if TARGET_COLUMN and TARGET_COLUMN not in df.columns: raise ValueError("TARGET_COLUMN not found")
if TARGET_COLUMN:
    n=df[TARGET_COLUMN].nunique(dropna=True)
    TASK_TYPE="classification" if (not pd.api.types.is_numeric_dtype(df[TARGET_COLUMN]) or n<=12) else "regression"
    print("Target:",TARGET_COLUMN,"Unique values:",n,"Task:",TASK_TYPE)
    display(df[TARGET_COLUMN].value_counts(dropna=False).head(20).to_frame("count"))
else:
    TASK_TYPE=None
    print("No obvious target found. Set TARGET_COLUMN manually to run ML.")

No obvious target found. Set TARGET_COLUMN manually to run ML.


## 15. Target distribution

Visualize class frequencies or a continuous target distribution.

In [15]:
if TARGET_COLUMN:
    if TASK_TYPE=="classification":
        v=df[TARGET_COLUMN].astype(str).value_counts().rename_axis("class").reset_index(name="count")
        if PLOTLY_AVAILABLE: px.bar(v,x="class",y="count",title="Target Class Distribution").show()
        else: v.plot.bar(x="class",y="count",legend=False); plt.show()
    else:
        if PLOTLY_AVAILABLE: px.histogram(df,x=TARGET_COLUMN,marginal="box",title="Target Distribution").show()
        else: df[TARGET_COLUMN].hist(); plt.show()
else: print("Target not set.")

Target not set.


## 16. Correlation heatmap

Inspect numeric correlations with an interactive heatmap.

In [16]:
# Numeric Correlation Heatmap
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    corr = corr.replace([np.inf, -np.inf], np.nan)

    # Remove columns with no valid correlations
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if corr.shape[0] >= 2 and corr.shape[1] >= 2:
        plotted = False

        # Try Plotly first
        if PLOTLY_AVAILABLE:
            try:
                fig = px.imshow(
                    corr,
                    text_auto=".2f",
                    aspect="auto",
                    color_continuous_scale="RdBu_r",
                    zmin=-1,
                    zmax=1,
                    title="Numeric Correlation Heatmap"
                )

                fig.update_layout(
                    width=1000,
                    height=max(600, len(corr) * 35),
                    template="plotly_white"
                )

                display(fig)
                plotted = True

            except Exception:
                print("Plotly rendering failed. Displaying Matplotlib heatmap.")

        # Matplotlib fallback
        if not plotted:
            fig, ax = plt.subplots(
                figsize=(max(10, len(corr) * 0.8), max(8, len(corr) * 0.65))
            )

            image = ax.imshow(
                corr,
                cmap="coolwarm",
                vmin=-1,
                vmax=1,
                aspect="auto"
            )

            fig.colorbar(image, ax=ax, label="Correlation")
            ax.set_xticks(range(len(corr.columns)))
            ax.set_xticklabels(corr.columns, rotation=90)
            ax.set_yticks(range(len(corr.index)))
            ax.set_yticklabels(corr.index)
            ax.set_title("Numeric Correlation Heatmap")

            # Add correlation values to cells
            for i in range(len(corr.index)):
                for j in range(len(corr.columns)):
                    value = corr.iloc[i, j]
                    if pd.notna(value):
                        ax.text(
                            j, i, f"{value:.2f}",
                            ha="center", va="center", fontsize=8
                        )

            plt.tight_layout()
            display(fig)
            plt.close(fig)

    else:
        print("Not enough valid numeric columns to calculate correlations.")

else:
    print("Need at least two numeric columns.")

## 17. Feature-versus-target plots

Compare numeric features against the chosen target.

In [17]:
if TARGET_COLUMN:
    for c in [x for x in num_cols if x!=TARGET_COLUMN][:8]:
        if PLOTLY_AVAILABLE:
            fig=px.box(df,x=TARGET_COLUMN,y=c,title=f"{c} by {TARGET_COLUMN}") if TASK_TYPE=="classification" else px.scatter(df,x=c,y=TARGET_COLUMN,title=f"{c} vs {TARGET_COLUMN}")
            fig.show()
        else:
            if TASK_TYPE=="classification": df.boxplot(column=c,by=TARGET_COLUMN,rot=30)
            else: plt.scatter(df[c],df[TARGET_COLUMN],alpha=.5)
            plt.tight_layout(); plt.show()
else: print("Target not set; skipping target comparison.")

Target not set; skipping target comparison.


## 18. Feature engineering

Create optional screen-time/sleep ratios and threshold indicators when relevant numeric columns exist.

In [18]:
def find_col(patterns):
    return next((c for c in df.columns if any(re.search(p,c,re.I) for p in patterns)),None)
screen=find_col([r"screen.*time",r"social.*media.*hour",r"usage.*hour"])
sleep=find_col([r"sleep.*hour",r"sleep.*duration"])
stress=find_col([r"stress.*score",r"stress.*level",r"perceived.*stress"])
engineered=[]
if screen and sleep and pd.api.types.is_numeric_dtype(df[screen]) and pd.api.types.is_numeric_dtype(df[sleep]):
    df["Screen_Time_to_Sleep_Ratio"]=df[screen]/df[sleep].replace(0,np.nan); engineered.append("Screen_Time_to_Sleep_Ratio")
if screen and pd.api.types.is_numeric_dtype(df[screen]):
    df["High_Screen_Time_Flag"]=(df[screen]>=df[screen].median()).astype(int); engineered.append("High_Screen_Time_Flag")
if stress and sleep and pd.api.types.is_numeric_dtype(df[stress]) and pd.api.types.is_numeric_dtype(df[sleep]):
    df["Stress_per_Sleep_Hour"]=df[stress]/df[sleep].replace(0,np.nan); engineered.append("Stress_per_Sleep_Hour")
print("Detected:",{"screen":screen,"sleep":sleep,"stress":stress})
print("Engineered:",engineered)

Detected: {'screen': 'Daily_Social_Media_Hours', 'sleep': 'Sleep_Hours', 'stress': None}
Engineered: ['Screen_Time_to_Sleep_Ratio', 'High_Screen_Time_Flag']


## 19. Prepare modeling table

Remove missing targets and obvious identifier/high-cardinality columns to reduce leakage risk.

In [19]:
if TARGET_COLUMN:
    model_df=df.dropna(subset=[TARGET_COLUMN]).copy()
    y=model_df[TARGET_COLUMN]
    X=model_df.drop(columns=[TARGET_COLUMN]).copy()
    id_cols=[c for c in X if re.search(r"(^id$|_id$|student.?id|record.?id|unnamed)",c,re.I) or X[c].nunique(dropna=True)>=max(100,int(.95*len(X)))]
    X=X.drop(columns=id_cols,errors="ignore")
    print("Model rows:",len(X),"Features:",X.shape[1],"Excluded ID-like columns:",id_cols)
else: print("Skipping model preparation because no target is selected.")

Skipping model preparation because no target is selected.


## 20. Train-validation split

Split rows into train and validation sets, stratifying classification if class counts permit.

In [20]:
if TARGET_COLUMN:
    from sklearn.model_selection import train_test_split
    if TASK_TYPE=="classification":
        y_model=y.astype(str)
        strat=y_model if y_model.value_counts().min()>=2 else None
    else:
        y_model=pd.to_numeric(y,errors="coerce")
        valid=y_model.notna(); X=X.loc[valid]; y_model=y_model.loc[valid]; strat=None
    X_train,X_valid,y_train,y_valid=train_test_split(X,y_model,test_size=.2,random_state=RANDOM_STATE,stratify=strat)
    print("Train:",X_train.shape,"Validation:",X_valid.shape)

## 21. Build preprocessing pipelines

Impute missing data, scale numeric features, and one-hot encode categoricals inside a leakage-aware scikit-learn pipeline.

In [21]:
if TARGET_COLUMN:
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder,StandardScaler
    from sklearn.dummy import DummyClassifier,DummyRegressor
    from sklearn.linear_model import LogisticRegression,Ridge
    from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,RandomForestRegressor,ExtraTreesRegressor
    numeric_features=X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_features=X_train.select_dtypes(exclude=np.number).columns.tolist()
    prep=ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric_features),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical_features)
    ])
    if TASK_TYPE=="classification":
        estimators={"Dummy Baseline":DummyClassifier(strategy="most_frequent"),"Logistic Regression":LogisticRegression(max_iter=1200,class_weight="balanced"),"Random Forest":RandomForestClassifier(n_estimators=200,min_samples_leaf=2,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1),"Extra Trees":ExtraTreesClassifier(n_estimators=200,min_samples_leaf=2,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1)}
    else:
        estimators={"Dummy Baseline":DummyRegressor(strategy="median"),"Ridge Regression":Ridge(),"Random Forest":RandomForestRegressor(n_estimators=180,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1),"Extra Trees":ExtraTreesRegressor(n_estimators=180,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1)}
    print("Numeric:",len(numeric_features),"Categorical:",len(categorical_features))

## 22. Train and compare models

Fit baseline and tree/linear models, then calculate suitable validation metrics.

In [22]:
if TARGET_COLUMN:
    from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,mean_absolute_error,mean_squared_error,r2_score
    trained={}; results=[]
    for name,est in estimators.items():
        pipe=Pipeline([("prep",prep),("model",est)])
        try:
            pipe.fit(X_train,y_train); pred=pipe.predict(X_valid); trained[name]=pipe
            if TASK_TYPE=="classification":
                row={"model":name,"accuracy":accuracy_score(y_valid,pred),"precision_weighted":precision_score(y_valid,pred,average="weighted",zero_division=0),"recall_weighted":recall_score(y_valid,pred,average="weighted",zero_division=0),"f1_weighted":f1_score(y_valid,pred,average="weighted",zero_division=0)}
                if len(pd.Series(y_valid).unique())==2 and hasattr(pipe,"predict_proba"):
                    try: row["roc_auc"]=roc_auc_score(y_valid,pipe.predict_proba(X_valid)[:,1])
                    except Exception: pass
            else: row={"model":name,"MAE":mean_absolute_error(y_valid,pred),"RMSE":mean_squared_error(y_valid,pred)**.5,"R2":r2_score(y_valid,pred)}
            results.append(row); print("Trained:",name)
        except Exception as e: print("Failed:",name,e)
    comparison=pd.DataFrame(results)
    display(comparison.sort_values("f1_weighted" if TASK_TYPE=="classification" else "RMSE",ascending=TASK_TYPE!="classification"))

## 23. Interactive model comparison

Compare model scores using Plotly.

In [23]:
if TARGET_COLUMN and not comparison.empty:
    metric="f1_weighted" if TASK_TYPE=="classification" else "RMSE"
    ordered=comparison.sort_values(metric,ascending=TASK_TYPE!="classification")
    if PLOTLY_AVAILABLE: px.bar(ordered,x="model",y=metric,title=f"Model Comparison: {metric}").show()
    else: ordered.plot.bar(x="model",y=metric,legend=False); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 24. Select the best model

Select the highest weighted F1 for classification or lowest RMSE for regression.

In [24]:
if TARGET_COLUMN and not comparison.empty:
    best_name=comparison.sort_values("f1_weighted",ascending=False).iloc[0]["model"] if TASK_TYPE=="classification" else comparison.sort_values("RMSE").iloc[0]["model"]
    best_model=trained[best_name]
    best_pred=best_model.predict(X_valid)
    print("Selected:",best_name)
else: print("No fitted model available.")

No fitted model available.


## 25. Classification evaluation or regression residuals

Display a classification report/confusion matrix or regression error plots.

In [25]:
if TARGET_COLUMN and "best_model" in globals():
    from sklearn.metrics import classification_report,confusion_matrix
    if TASK_TYPE=="classification":
        print(classification_report(y_valid,best_pred,zero_division=0))
        labels=sorted(pd.Series(y_valid).astype(str).unique()); cm=confusion_matrix(y_valid,best_pred,labels=labels)
        if PLOTLY_AVAILABLE: px.imshow(cm,x=labels,y=labels,text_auto=True,title="Validation Confusion Matrix").show()
        else: plt.imshow(cm); plt.colorbar(); plt.show()
    else:
        residual=y_valid.to_numpy()-np.asarray(best_pred)
        if PLOTLY_AVAILABLE:
            px.scatter(x=y_valid,y=best_pred,labels={"x":"Actual","y":"Predicted"},title="Actual vs Predicted").show()
            px.histogram(x=residual,nbins=30,title="Residual Distribution").show()
        else:
            plt.scatter(y_valid,best_pred); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.show()
            plt.hist(residual,bins=30); plt.show()

## 26. Feature importance

Extract tree feature importance or linear coefficient magnitude when supported.

In [26]:
if TARGET_COLUMN and "best_model" in globals():
    model=best_model.named_steps["model"]
    try:
        names=best_model.named_steps["prep"].get_feature_names_out()
        if hasattr(model,"feature_importances_"): vals=model.feature_importances_
        elif hasattr(model,"coef_"):
            co=np.asarray(model.coef_); vals=np.mean(np.abs(co),axis=0) if co.ndim>1 else np.abs(co)
        else: vals=None
        if vals is not None:
            importance=pd.DataFrame({"feature":names,"importance":vals}).sort_values("importance",ascending=False).head(25)
            display(importance)
            if PLOTLY_AVAILABLE: px.bar(importance.sort_values("importance"),x="importance",y="feature",orientation="h",title="Top Features").show()
    except Exception as e: print("Importance unavailable:",e)

## 27. Examine class balance

Report target balance for classification or target range for regression.

In [27]:
if TARGET_COLUMN:
    if TASK_TYPE=="classification":
        balance=y.value_counts(normalize=True).mul(100).round(2).rename_axis("class").reset_index(name="percent")
        display(balance)
    else:
        print("Mean:",pd.to_numeric(y).mean(),"Std:",pd.to_numeric(y).std(),"Min:",pd.to_numeric(y).min(),"Max:",pd.to_numeric(y).max())

## 28. Subgroup comparison

Compare target outcomes across a low-cardinality categorical variable, if available.

In [28]:
if TARGET_COLUMN:
    groups=[c for c in df.select_dtypes(exclude=np.number).columns if c!=TARGET_COLUMN and df[c].nunique()<=15]
    if groups:
        g=groups[0]; print("Group column:",g)
        if TASK_TYPE=="classification":
            tab=pd.crosstab(df[g].fillna("Missing"),df[TARGET_COLUMN].fillna("Missing"),normalize="index")*100
            display(tab.round(2))
            if PLOTLY_AVAILABLE: px.bar(tab.reset_index().melt(id_vars=g,var_name="target",value_name="percent"),x=g,y="percent",color="target",barmode="stack",title=f"Target composition by {g}").show()
        else: display(df.groupby(g)[TARGET_COLUMN].agg(["count","mean","median"]).sort_values("count",ascending=False))
    else: print("No suitable categorical subgroup found.")

## 29. Save model comparison and cleaned data

Export reproducible CSV outputs into Kaggle's working directory.

In [29]:
OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
df.to_csv(OUT/"cleaned_student_social_media_data.csv",index=False)
if "comparison" in globals() and not comparison.empty: comparison.to_csv(OUT/"model_comparison.csv",index=False)
if "importance" in globals() and not importance.empty: importance.to_csv(OUT/"feature_importance.csv",index=False)
print("Output files:")
for p in OUT.glob("*.csv"): print(p.name,round(p.stat().st_size/1024,1),"KB")

Output files:
cleaned_student_social_media_data.csv 1267.3 KB


## 30. Generate data-driven summary

Summarize dataset quality and model performance without fabricating results.

In [30]:
print("DATASET SUMMARY")
print("Rows:",len(df),"Columns:",df.shape[1])
print("Numeric columns:",len(df.select_dtypes(include=np.number).columns))
print("Categorical columns:",len(df.select_dtypes(exclude=np.number).columns))
print("Missing cells:",int(df.isna().sum().sum()))
print("Duplicate rows:",int(df.duplicated().sum()))
print("Target:",TARGET_COLUMN)
if TARGET_COLUMN and "best_name" in globals():
    row=comparison[comparison.model==best_name].iloc[0]
    print("Best model:",best_name)
    print(row.to_dict())

DATASET SUMMARY
Rows: 16000 Columns: 12
Numeric columns: 9
Categorical columns: 3
Missing cells: 0
Duplicate rows: 0
Target: None


## 31. Responsible interpretation

Document the limits of the analysis and responsible use of student-health information.

In [31]:
notes=[
"Confirm the target definition and feature meanings against the dataset documentation.",
"Correlation and subgroup differences do not prove that social-media use causes health or grade outcomes.",
"Survey or synthetic data may not represent all student populations.",
"Validation metrics depend on target quality, class balance, and the held-out split.",
"Do not use predictions as medical diagnoses or to penalize individual students."
]
for note in notes: print("-",note)

- Confirm the target definition and feature meanings against the dataset documentation.
- Correlation and subgroup differences do not prove that social-media use causes health or grade outcomes.
- Survey or synthetic data may not represent all student populations.
- Validation metrics depend on target quality, class balance, and the held-out split.
- Do not use predictions as medical diagnoses or to penalize individual students.


## 32. Final checklist

Confirm that the main stages were attempted and state what was exported.

In [32]:
checklist={
"Dataset loaded": "df" in globals(),
"Basic cleaning completed": "df" in globals(),
"Interactive visualization available": PLOTLY_AVAILABLE,
"Target selected": TARGET_COLUMN is not None,
"Model comparison created": "comparison" in globals() and not comparison.empty if "comparison" in globals() else False,
"Cleaned CSV exported": (Path("/kaggle/working")/"cleaned_student_social_media_data.csv").exists()
}
for k,v in checklist.items(): print(("PASS" if v else "CHECK"),k)
print("\nPipeline finished. Review the selected target and validation results before drawing conclusions.")

PASS Dataset loaded
PASS Basic cleaning completed
PASS Interactive visualization available
CHECK Target selected
CHECK Model comparison created
PASS Cleaned CSV exported

Pipeline finished. Review the selected target and validation results before drawing conclusions.
